# 09 — Dashboard operativo de Sentry con explicabilidad SHAP

**Proyecto:** VIGÍA (Sentry) — TFC
**Objetivo:** construir una interfaz operativa que muestre al analista de SOC:
1. Alertas ordenadas por confianza del modelo.
2. Familia predicha (BENIGN, DoS, DDoS, etc.).
3. Explicación SHAP: top-3 features que justifican la predicción.
4. Filtro por familia y búsqueda.

**Motivo de seguridad:** una alerta sin contexto no es accionable. Este dashboard cierra el ciclo detección → explicación → decisión.

**Alcance:** versión funcional con Streamlit, ejecutada en Colab mediante túnel `ngrok`. Las capturas de pantalla se usan en la memoria y el vídeo.

In [ ]:
# ============================================
# Setup del notebook 09
# ============================================
import os
from google.colab import drive
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when
from pyspark.sql.types import NumericType
import numpy as np
import xgboost as xgb
import joblib
import gc

GITHUB_USER = "AguCS231"
REPO_NAME   = "sentry"
REPO_LOCAL  = f"/content/{REPO_NAME}"
BASE_SPLITS = "/content/drive/MyDrive/cic-ids2017-splits"

if not os.path.exists(REPO_LOCAL):
    %cd /content
    !git clone https://github.com/{GITHUB_USER}/{REPO_NAME}.git
    %cd {REPO_LOCAL}
else:
    %cd {REPO_LOCAL}

try:
    drive.mount('/content/drive')
except:
    pass

try:
    spark
except NameError:
    spark = SparkSession.builder \
        .appName("VIGIA_Dashboard") \
        .config("spark.driver.memory", "4g") \
        .getOrCreate()

print(f"✅ Spark {spark.version}")

In [ ]:
# ============================================
# Cargar modelo oficial + muestra de test con etiquetas
# ============================================
import time

MODELS_DIR = f"{REPO_LOCAL}/src/models"

# Modelo oficial (familias sin Destination_Port)
modelo = xgb.XGBClassifier()
modelo.load_model(f"{MODELS_DIR}/familias_sin_puerto.json")
print("✅ Modelo cargado")

# Mapeo
familias = ['BENIGN', 'DoS', 'Probe', 'DDoS', 'Brute-force', 'Web', 'Bot']
mapeo = {fam: i for i, fam in enumerate(familias)}

# Cargar test y muestrear 500 alertas
test_df = spark.read.parquet(f"{BASE_SPLITS}/test.parquet")

expr = None
for fam, idx in mapeo.items():
    cond = when(col("target_family") == fam, idx)
    expr = cond if expr is None else expr.when(col("target_family") == fam, idx)
test_df = test_df.withColumn("target_family_idx", expr.otherwise(-1))

# Excluir "Otros" para el dashboard
test_eval = test_df.filter("target_family_idx >= 0")

# Features
excluir = {"Label", "target_bin", "target_family", "Destination_Port", "target_family_idx"}
feature_cols = [
    c for c in test_eval.columns
    if c not in excluir and isinstance(test_eval.schema[c].dataType, NumericType)
]
print(f"Features: {len(feature_cols)}")

# Muestra de 500 flujos
print("\nMuestreando 500 flujos...")
t0 = time.time()
sample = test_eval.sample(fraction=0.002, seed=42).limit(500)
sample_pd = sample.select(feature_cols + ["target_family_idx", "Label"]).toPandas()
print(f"✅ Muestra: {sample_pd.shape} en {time.time()-t0:.1f}s")

# Guardar muestra en CSV para que Streamlit la lea
sample_pd.to_csv(f"{REPO_LOCAL}/app/muestra_alertas.csv", index=False)
print(f"✅ Guardado en app/muestra_alertas.csv")

In [ ]:
# ============================================
# Instalar dependencias del dashboard
# ============================================
!pip install streamlit shap pyngrok -q

print("✅ Dependencias instaladas")

In [ ]:
# ============================================
# Crear app/dashboard.py
# ============================================
dashboard_code = '''
"""
Sentry — Dashboard operativo de alertas
Panel interactivo para analistas de SOC.

Ejecutar: streamlit run app/dashboard.py
"""
import streamlit as st
import pandas as pd
import numpy as np
import xgboost as xgb
import shap
import json
import os

# ============================================
# Configuración
# ============================================
st.set_page_config(page_title="Sentry", page_icon="🛡️", layout="wide")

BASE_DIR = os.path.dirname(os.path.abspath(__file__))
REPO_DIR = os.path.dirname(BASE_DIR)
MODELS_DIR = os.path.join(REPO_DIR, "src", "models")

FAMILIAS = [\'BENIGN\', \'DoS\', \'Probe\', \'DDoS\', \'Brute-force\', \'Web\', \'Bot\']

# ============================================
# Cargar modelo y datos (cacheado)
# ============================================
@st.cache_resource
def cargar_modelo():
    modelo = xgb.XGBClassifier()
    modelo.load_model(os.path.join(MODELS_DIR, "familias_sin_puerto.json"))
    return modelo

@st.cache_data
def cargar_muestra():
    df = pd.read_csv(os.path.join(BASE_DIR, "muestra_alertas.csv"))
    return df

@st.cache_resource
def cargar_explainer(_modelo):
    return shap.TreeExplainer(_modelo)

# ============================================
# Encabezado
# ============================================
st.title("🛡️ Sentry — Panel operativo de detección de intrusiones")
st.caption("Alertas clasificadas por familia con explicación SHAP")

# ============================================
# Carga de datos
# ============================================
modelo = cargar_modelo()
df = cargar_muestra()

# Excluir columnas que no son features
cols_no_features = ["target_family_idx", "Label"]
feature_cols = [c for c in df.columns if c not in cols_no_features]

# Predicciones
X = df[feature_cols].values.astype(np.float32)
probs = modelo.predict_proba(X)
preds = probs.argmax(axis=1)
confianza = probs.max(axis=1)

df["prediccion_idx"] = preds
df["prediccion_familia"] = [FAMILIAS[p] for p in preds]
df["confianza"] = confianza
df["real_idx"] = df["target_family_idx"]
df["real_familia"] = [FAMILIAS[int(i)] for i in df["real_idx"]]

# ============================================
# Filtros
# ============================================
st.sidebar.header("Filtros")
familias_disponibles = ["Todas"] + FAMILIAS
filtro_familia = st.sidebar.selectbox("Familia predicha", familias_disponibles)
filtro_confianza = st.sidebar.slider("Confianza mínima", 0.0, 1.0, 0.5, 0.05)
solo_ataques = st.sidebar.checkbox("Solo ataques (excluir BENIGN)", value=True)
solo_errores = st.sidebar.checkbox("Solo errores del modelo", value=False)

df_filtrado = df[df["confianza"] >= filtro_confianza]
if filtro_familia != "Todas":
    df_filtrado = df_filtrado[df_filtrado["prediccion_familia"] == filtro_familia]
if solo_ataques:
    df_filtrado = df_filtrado[df_filtrado["prediccion_familia"] != "BENIGN"]
if solo_errores:
    df_filtrado = df_filtrado[df_filtrado["prediccion_idx"] != df_filtrado["real_idx"]]

df_filtrado = df_filtrado.sort_values("confianza", ascending=False)

# ============================================
# Métricas resumen
# ============================================
col1, col2, col3, col4 = st.columns(4)
col1.metric("Alertas totales", len(df))
col2.metric("Alertas filtradas", len(df_filtrado))
col3.metric("Ataques detectados", (df["prediccion_familia"] != "BENIGN").sum())
col4.metric("Confianza media", f"{df['confianza'].mean():.2%}")

# ============================================
# Distribución por familia
# ============================================
st.subheader("Distribución por familia predicha")
dist = df["prediccion_familia"].value_counts()
st.bar_chart(dist)

# ============================================
# Lista de alertas con explicación SHAP
# ============================================
st.subheader(f"Alertas ({len(df_filtrado)})")

if len(df_filtrado) == 0:
    st.info("No hay alertas que cumplan los filtros.")
else:
    # Inicializar explainer solo cuando haga falta (es costoso)
    explainer = cargar_explainer(modelo)

    # Mostrar hasta 20 alertas
    for idx, row in df_filtrado.head(20).iterrows():
        with st.expander(
            f"🔔 [{row['prediccion_familia']}] confianza {row['confianza']:.2%}  |  real: {row['real_familia']}",
            expanded=False
        ):
            col_a, col_b = st.columns([2, 1])

            with col_a:
                # Calcular SHAP para este flujo
                x_uno = df.loc[[idx], feature_cols].values.astype(np.float32)
                shap_vals = explainer.shap_values(x_uno)
                if isinstance(shap_vals, list):
                    shap_arr = np.stack(shap_vals, axis=-1)
                else:
                    shap_arr = shap_vals

                clase_pred = int(row["prediccion_idx"])
                contribuciones = shap_arr[0, :, clase_pred]
                top3_idx = np.argsort(np.abs(contribuciones))[::-1][:3]

                st.markdown("**Top 3 features que justifican la predicción:**")
                for i in top3_idx:
                    feat = feature_cols[i]
                    val = contribuciones[i]
                    color = "🔴" if val > 0 else "🟢"
                    st.markdown(f"- {color} `{feat}`: contribución {val:+.4f}")

            with col_b:
                st.markdown("**Confianza por clase:**")
                for fam, p in zip(FAMILIAS, probs[idx]):
                    st.progress(float(p), text=f"{fam}: {p:.2%}")
'''

with open(f"{REPO_LOCAL}/app/dashboard.py", "w") as f:
    f.write(dashboard_code)

print("✅ app/dashboard.py creado")
!ls -lh {REPO_LOCAL}/app/

In [ ]:
# ============================================
# Lanzar Streamlit + túnel ngrok
# ============================================
import os
import time
from pyngrok import ngrok, conf
from getpass import getpass

# 1. Pedir token de ngrok

NGROK_TOKEN = getpass("Token de ngrok: ")
conf.get_default().auth_token = NGROK_TOKEN

# 2. Matar túneles previos si existen
try:
    ngrok.kill()
except:
    pass

# 3. Lanzar Streamlit en background
os.chdir(f"{REPO_LOCAL}/app")
get_ipython().system_raw(
    "streamlit run dashboard.py --server.port 8501 --server.headless true > /tmp/streamlit.log 2>&1 &"
)
time.sleep(5)

# 4. Abrir túnel
url = ngrok.connect(8501, "http")
print(f"\n✅ Dashboard disponible en: {url}")
print(f"\nSi el enlace no carga, espera 10 segundos y recarga.")

In [ ]:
# Ver los logs de Streamlit
!cat /tmp/streamlit.log | tail -50

In [ ]:
# ============================================
# Matar Streamlit y ngrok previos
# ============================================
!pkill -f streamlit
!pkill -f ngrok
import time
time.sleep(3)

# Verificar que ya no hay nada en el puerto
!lsof -i :8501 || echo "✅ Puerto 8501 libre"

In [ ]:
# ============================================
# Relanzar Streamlit + ngrok
# ============================================
import os
import time
from pyngrok import ngrok, conf
from getpass import getpass

# Token de ngrok
NGROK_TOKEN = getpass("Token de ngrok: ")
conf.get_default().auth_token = NGROK_TOKEN

# Limpiar ngrok por si acaso
try:
    ngrok.kill()
except:
    pass

# Limpiar log anterior
!rm -f /tmp/streamlit.log

# Lanzar Streamlit
os.chdir(f"/content/sentry/app")
get_ipython().system_raw(
    "streamlit run dashboard.py --server.port 8501 --server.headless true > /tmp/streamlit.log 2>&1 &"
)

# Esperar a que arranque de verdad
print("Esperando a que Streamlit arranque...")
time.sleep(10)

# Comprobar el log
print("\n=== Log de Streamlit ===")
!cat /tmp/streamlit.log

# Abrir túnel
try:
    url = ngrok.connect(8501, "http")
    print(f"\n✅ Dashboard disponible en: {url}")
    print("\nAbre el enlace en el navegador. Si sigue en blanco, espera 10s y recarga.")
except Exception as e:
    print(f"\n❌ Error al conectar ngrok: {e}")

In [ ]:
# ============================================
# Obtener la URL actual del túnel ngrok
# ============================================
from pyngrok import ngrok

tunnels = ngrok.get_tunnels()
if tunnels:
    for t in tunnels:
        print(f"🔗 URL del dashboard: {t.public_url}")
else:
    print("⚠️ No hay túneles activos. Relanza el túnel con:")
    print("   ngrok.connect(8501, 'http')")

In [ ]:
# ============================================
# Regenerar muestra con muestreo estratificado
# ============================================
import time
from functools import reduce
from pyspark.sql import DataFrame

print("Muestreando por familias (estratificado)...")

# Cuotas por familia para un dashboard útil
cuotas = {
    'BENIGN': 200,      # la mayoritaria con más peso
    'DoS': 80,
    'Probe': 80,
    'DDoS': 80,
    'Brute-force': 30,
    'Web': 20,
    'Bot': 20,
}
# Total: 510 alertas

partes = []
for fam, n in cuotas.items():
    df_fam = test_eval.filter(col("target_family") == fam)
    total_fam = df_fam.count()
    frac = min(n / total_fam, 1.0)  # evitar fracción > 1
    muestra_fam = df_fam.sample(fraction=frac, seed=42).limit(n)
    partes.append(muestra_fam)
    print(f"  {fam}: {n} pedidas → {muestra_fam.count()} obtenidas")

sample = reduce(DataFrame.unionByName, partes)
print(f"\nTotal muestra: {sample.count()}")

# Convertir a pandas y guardar
sample_pd = sample.select(feature_cols + ["target_family_idx", "Label"]).toPandas()
sample_pd = sample_pd.reset_index(drop=True)

print(f"\nDistribución real de la muestra:")
print(sample_pd["Label"].value_counts())

# Guardar CSV
sample_pd.to_csv(f"{REPO_LOCAL}/app/muestra_alertas.csv", index=False)
print(f"\n✅ Guardado: {len(sample_pd)} filas × {len(sample_pd.columns)} columnas")

In [ ]:
# ============================================
# Matar Streamlit y relanzar (limpia caché)
# ============================================
!pkill -f streamlit
import time
time.sleep(3)

# Verificar el CSV antes de relanzar
import pandas as pd
df_check = pd.read_csv("/content/sentry/app/muestra_alertas.csv")
print("=== Distribución del CSV ===")
print(df_check["Label"].value_counts())
print(f"\nTotal: {len(df_check)}")

# Relanzar Streamlit
import os
os.chdir("/content/sentry/app")
get_ipython().system_raw(
    "streamlit run dashboard.py --server.port 8501 --server.headless true > /tmp/streamlit.log 2>&1 &"
)
time.sleep(8)
print("\n✅ Streamlit relanzado")
!tail -10 /tmp/streamlit.log

In [ ]:
%cd /content/sentry
!mkdir -p docs/screenshots
!git add .
!git commit -m "Dashboard operativo: capturas del panel con SHAP integrado"


In [ ]:
from getpass import getpass
TOKEN = getpass("Pega tu token: ")
%cd /content/sentry
!git remote set-url origin https://{TOKEN}@github.com/AguCS231/sentry.git
!git push